# 🕵️ Dia 32 — OPSEC do investigador: VPN, Tor, isolamento e teste de vazamentos

**Investigador Cyber 90 Dias** · Fase 2 · Rank Jutsu `C`

> Baseado no *Day 32: Investigator OPSEC: VPN, Tor, isolation, and leak testing* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-2-osint-digital-footprint/day-32.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Terça, 13 de outubro. Desde o dia 6, quem abre o site falso da rede da cooperativa (ou da Vigia) vê uma **página em branco**. Rafa mostra um log **ilustrativo** do tipo que o servidor do kit grava, e uma das linhas é sua.

*"Hoje você descobre o que cada clique seu conta para o outro lado."*

Neste notebook você vai:

1. **Ver o rastro do próprio Colab** (IP de saída, fuso, idioma), com plano B.
2. **Ler o log ilustrativo do kit** e achar o que o operador provavelmente achou.
3. **Montar o mapa de calor** das quatro configurações.
4. **Gerar a norma de OPSEC** de uma página.
5. **Jogar o quiz "qual configuração?"**.

🐍 Só biblioteca padrão.

## 1. O pré-voo: o que esta máquina mostra ao mundo?

No Colab, o "você" é uma máquina do Google. Rode mesmo assim: o objetivo é ver **quais campos** qualquer site recebe.

In [ ]:
import json, time, locale, urllib.request
from datetime import datetime

try:
    with urllib.request.urlopen("https://check.torproject.org/api/ip", timeout=10) as r:
        ip = json.load(r)                                   # {"IsTor": false, "IP": "..."}
    origem = "check.torproject.org (ao vivo)"
except Exception as e:                                       # sem rede: amostra com a mesma estrutura
    ip = {"IsTor": False, "IP": "198.51.100.23"}             # IP de documentação, fictício
    origem = f"AMOSTRA EMBUTIDA ({type(e).__name__})"

print("Fonte:", origem)
print("IP que os sites veem:", ip["IP"], "| saindo pelo Tor?", ip["IsTor"])
print("Fuso horário anunciado:", time.tzname, "| offset UTC:", -time.timezone // 3600, "h")
print("Idioma/locale:", locale.getlocale())
print("Hora local agora:", datetime.now().strftime("%H:%M"),
      "-> horário comercial também é um sinal (Conceito do original).")

In [ ]:
# OPCIONAL: o Passo 1 do original pelo terminal (o teste pula linhas com !)
!curl -s https://ipinfo.io/json
!curl -s https://check.torproject.org/api/ip
print("(fora do Colab, as linhas com ! foram puladas)")

## 2. O que o kit viu (log ilustrativo)

Este log é **fictício e ilustrativo**: a Vigia não tem os logs reais do servidor do golpe (eles só saem por ordem judicial). Ele foi reconstruído para mostrar o que o operador provavelmente enxerga. Formato: `data_utc | ip | asn_dono | user_agent | accept_language | fuso_js | referer | resposta`.

In [ ]:
LOG = r"""# Log ilustrativo (fictício) do tipo que o servidor do IscaKit grava. A Vigia NÃO tem este log:
# ele mostra o que o "Pescador" provavelmente vê de quem visita o site falso.
# formato: data_utc | ip | asn_dono | user_agent | accept_language | fuso_js | referer | resposta
2026-10-05T10:31:02Z | 198.51.100.4    | AS64503 OPERADORA-MOVEL-EXEMPLO | Mozilla/5.0 (Linux; Android 14) Mobile | pt-BR | America/Sao_Paulo | lnk.example/sa7 | 200 kit
2026-10-05T10:31:40Z | 198.51.100.9    | AS64503 OPERADORA-MOVEL-EXEMPLO | Mozilla/5.0 (iPhone; CPU iPhone OS 18_0) | pt-BR | America/Sao_Paulo | lnk.example/sa7 | 200 kit
2026-10-05T12:14:07Z | 198.51.100.130  | AS64502 COOPERATIVA-SERRA-AZUL  | Mozilla/5.0 (Windows NT 10.0) Chrome | pt-BR | America/Sao_Paulo | (e-mail interno) | 200 kit
2026-10-05T14:02:55Z | 192.0.2.40      | AS64504 VIGIA-ESCRITORIO        | Mozilla/5.0 (X11; Linux x86_64) Firefox | pt-BR,en | America/Sao_Paulo | (direto) | 200 kit
2026-10-05T14:03:10Z | 192.0.2.40      | AS64504 VIGIA-ESCRITORIO        | Mozilla/5.0 (X11; Linux x86_64) Firefox | pt-BR,en | America/Sao_Paulo | (direto) | 200 kit
2026-10-06T09:10:12Z | 198.51.100.130  | AS64502 COOPERATIVA-SERRA-AZUL  | Mozilla/5.0 (Windows NT 10.0) Chrome | pt-BR | America/Sao_Paulo | (direto) | 200 pagina-em-branco
2026-10-06T09:40:31Z | 192.0.2.40      | AS64504 VIGIA-ESCRITORIO        | Mozilla/5.0 (X11; Linux x86_64) Firefox | pt-BR,en | America/Sao_Paulo | (direto) | 200 pagina-em-branco
2026-10-06T09:52:44Z | 203.0.113.200   | AS64505 VPN-COMERCIAL-EXEMPLO   | Mozilla/5.0 (X11; Linux x86_64) Firefox | pt-BR | America/Sao_Paulo | (direto) | 200 kit
2026-10-06T09:55:03Z | 198.51.100.15   | AS64503 OPERADORA-MOVEL-EXEMPLO | Mozilla/5.0 (Linux; Android 13) Mobile | pt-BR | America/Sao_Paulo | lnk.example/sa7 | 200 kit
"""

visitas = []
for linha in LOG.splitlines():
    if not linha.strip() or linha.startswith("#"):
        continue
    campos = [c.strip() for c in linha.split("|")]
    visitas.append(dict(zip(["data", "ip", "asn", "ua", "lang", "tz", "referer", "resp"], campos)))

print(f"{len(visitas)} visitas")
for v in visitas:
    print(f"{v['data']}  {v['asn'][:32]:32s} {v['ua'][:28]:28s} ref={v['referer']:16s} -> {v['resp']}")

### Pense como o operador

Uma vítima típica chega **pelo encurtador do SMS**, de **operadora móvel**, com **celular**. Vamos dar 1 ponto de "estranheza" para cada sinal fora desse perfil e ver quem se destaca.

In [ ]:
def estranheza(v):
    """Quantos sinais fogem do perfil 'cooperado que clicou no SMS'. Devolve (pontos, motivos)."""
    motivos = []
    if "lnk.example" not in v["referer"]:
        motivos.append("sem referer do SMS")
    if "MOVEL" not in v["asn"]:
        motivos.append(f"rede não móvel ({v['asn'].split()[1]})")
    if "Mobile" not in v["ua"] and "iPhone" not in v["ua"]:
        motivos.append("navegador de desktop")
    if "en" in v["lang"]:
        motivos.append("idioma inclui inglês")
    return len(motivos), motivos


for v in visitas:
    pontos, motivos = estranheza(v)
    barra = "█" * pontos + "·" * (4 - pontos)
    print(f"{v['data']}  {barra}  {v['resp']:16s} {', '.join(motivos) or 'perfil de vítima'}")

print("\nLeitura: as linhas com 3-4 pontos (cooperativa e Vigia) são exatamente as que, a partir de 6/out,")
print("passam a receber 'pagina-em-branco'. A da VPN recebe o kit: o IP mudou, mas o navegador Linux")
print("e o fuso continuam iguais aos da Vigia. É o 'cloaking' da Camada 8.")

## 3. O mapa de calor das quatro configurações

Preencha com os resultados dos **seus** testes (Passos 1 a 5). Escala: 0 = sem vazamento/coerente, 1 = parcial, 2 = vaza a identidade real. Os valores padrão são os do exemplo do original.

In [ ]:
#@title Resultados dos testes { display-mode: "form" }
A = "2,2,2,2,0,2"  #@param {type:"string"}
B = "0,2,0,2,2,0"  #@param {type:"string"}
C = "0,0,0,0,0,0"  #@param {type:"string"}
D = "0,0,0,1,0,0"  #@param {type:"string"}

CHECAGENS = ["IP real", "WebRTC", "DNS", "Impressão", "Fuso", "Sessões"]
CONFIGS = {"A dia a dia": A, "B perfil+VPN": B, "C Tor Browser": C, "D VM+perfil+VPN": D}
COR = {0: "🟩", 1: "🟨", 2: "🟥"}

print(f"{'Config':18s}" + "".join(f"{c:>11s}" for c in CHECAGENS))
pendentes = []
for nome, valores in CONFIGS.items():
    nums = [int(x) for x in valores.split(",")]
    print(f"{nome:18s}" + "".join(f"{COR[n]:>10s} " for n in nums))
    pendentes += [(nome, CHECAGENS[i], n) for i, n in enumerate(nums) if n > 0]

print(f"\n{len(pendentes)} células vermelhas/âmbar. Cada uma precisa de mitigação E novo teste (Passo 8):")
for nome, chec, n in pendentes:
    print(f"  - {nome} / {chec} ({'vermelho' if n == 2 else 'âmbar'}): mitigação = ______  novo teste = ______")

## 4. 📝 A norma de OPSEC de uma página

In [ ]:
#@title Norma de OPSEC { display-mode: "form" }
passiva_org = "B ou D"  #@param ["A", "B ou D", "C", "D"]
site_golpe = "D"  #@param ["A", "B", "C", "D"]
login_persona = "D (só com política de persona; Dia 31)"  #@param {type:"string"}
encurtador_sms = "urlscan.io Private; nunca clicar"  #@param {type:"string"}

norma = f"""# Norma de OPSEC — Vigia / Operação Isca

| Tarefa | Configuração | Por quê | Nunca |
|---|---|---|---|
| Link suspeito do e-mail de uma vítima | urlscan.io Private; se preciso, C ou D | o link pode ser único da vítima e registrar a visita | clicar no cliente de e-mail; urlscan Public |
| Navegação passiva no site de uma organização pública | {passiva_org} | separa cookies e sessões da vida real | logado em conta pessoal |
| Visita a site suspeito de golpe | {site_golpe} | o operador lê o log: rede, navegador e fuso precisam parecer de vítima comum | rede do escritório ou da cliente |
| Plataforma com login, com persona | {login_persona} | persona só existe dentro de compartimento | conta pessoal no mesmo perfil |
| ➕ Encurtador do SMS (lnk.example/sa7) | {encurtador_sms} | encurtadores contam cada clique | abrir da rede da cooperativa |

## Regras permanentes
- Links suspeitos: urlscan.io Private, ou C/D. Nunca A.
- E-mail suspeito: salvar o .eml e analisar offline. Nada de clicar ou carregar imagens.
- Nunca conta pessoal em B, C ou D.
- Para analisadores públicos vai o hash, não o arquivo, salvo política que permita upload.
"""
open("norma-opsec.md", "w", encoding="utf-8").write(norma)
print(norma)
if site_golpe in ("A", "B"):
    print("⚠️ Reveja: o log do Exercício 1 mostra o que A e B contaram ao kit.")

## 5. 🎮 Quiz: qual configuração?

Responda **A**, **B**, **C** ou **D** (ou **U** para "urlscan Private, sem abrir"). `MODO_JOGO = False` mostra só o gabarito.

In [ ]:
QUIZ = [
    ("Ler a página 'Canais oficiais' de serraazul.example (o cliente).", "B",
     "Passivo e público. B ou D servem; o importante é não estar logado em contas pessoais."),
    ("Capturar o formulário de serraazul-pix.example para o relatório.", "D",
     "Site do alvo: isolamento total e rede que não seja da Vigia nem da cooperativa."),
    ("Um cooperado encaminhou o SMS com lnk.example/sa7. Ver para onde vai.", "U",
     "Encurtador conta cliques. Primeiro urlscan Private."),
    ("Ver o perfil público @serraazul.oficial sem login.", "D",
     "É conteúdo do alvo: mesmo cuidado do site do golpe."),
    ("Conferir se o seu IP de saída mudou depois de ligar a VPN.", "B",
     "check.torproject.org/api/ip no próprio compartimento que você vai usar."),
    ("Abrir um anexo .pdf de um e-mail de golpe para ler os metadados.", "D",
     "Offline, dentro da VM; nunca no computador do dia a dia."),
    ("Ler o Marco Civil no planalto.gov.br.", "A",
     "Fonte pública sem relação com o alvo: o dia a dia serve."),
    ("Acessar um site que bloqueia Tor, ligado ao caso, para uma captura.", "D",
     "Tor seria bloqueado e ainda é um sinal; VM + VPN coerente com a lenda."),
]

MODO_JOGO = True
acertos = 0
for i, (tarefa, certo, porque) in enumerate(QUIZ, 1):
    print(f"\n[{i}/{len(QUIZ)}] {tarefa}")
    if MODO_JOGO:
        r = input("   A / B / C / D / U: ").strip().upper()
        acertos += (r == certo)
        print("   ✅" if r == certo else "   ❌", end=" ")
    print(f"Gabarito: {certo}. {porque}")
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)} — meta do Checkpoint BR: 6.")

## ✅ Antes de fechar o Dia 32

- [ ] Rastro do pré-voo anotado (IP, fuso, idioma)
- [ ] Linhas da Vigia no log ilustrativo identificadas, com os sinais
- [ ] Mapa de calor com mitigação e novo teste em cada célula vermelha/âmbar
- [ ] `norma-opsec.md` com as cinco tarefas
- [ ] Risco residual em duas frases
- [ ] Quiz com pelo menos 6/8

**Amanhã (Dia 33):** a lei. Computador, contrato e privacidade são três perguntas diferentes, e o Dr. Hélio quer a árvore de decisão antes da próxima coleta.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>